# Classical Machine Learning Models for Behavioral Fingerprinting
This notebook trains, evaluates, and compares classical ML classifiers for keystroke dynamics subject identification.
To avoid session leakage, the evaluation strictly uses a session-based train/test split (Sessions 1-6 for training, Sessions 7-8 for testing).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src module is discoverable
sys.path.insert(0, os.path.abspath('..'))

from src.config import ALL_FEATURE_COLUMNS, TRAIN_SESSIONS, TEST_SESSIONS, RANDOM_SEED
from src.preprocessing import load_raw_dataset, session_based_split, fit_scaler, transform_features, fit_label_encoder
from src.features import extract_features
from src.models import train_and_compare_all
from src.evaluation import compute_classification_metrics


## 1. Load Data and Extract Behavioral Features


In [ ]:
# Load cleaned raw dataset
df_raw = load_raw_dataset('../data/raw/DSL-StrongPasswordData.csv')
print(f'Loaded raw dataset: {df_raw.shape[0]} rows, {df_raw["subject"].nunique()} subjects')

# Extract 17 engineered features (total 48 features)
df_features = extract_features(df_raw)
print(f'Extracted features dataset shape: {df_features.shape}')
df_features.head()


## 2. Session-Based Train-Test Split (Zero Session Leakage)


In [ ]:
train_df, test_df = session_based_split(
    df_features, train_sessions=TRAIN_SESSIONS, test_sessions=TEST_SESSIONS
)

X_train_df = train_df[ALL_FEATURE_COLUMNS]
y_train_raw = train_df['subject'].values
X_test_df = test_df[ALL_FEATURE_COLUMNS]
y_test_raw = test_df['subject'].values

# Fit scaler and label encoder on training data only
scaler = fit_scaler(X_train_df)
X_train_scaled = transform_features(scaler, X_train_df, ALL_FEATURE_COLUMNS)
X_test_scaled = transform_features(scaler, X_test_df, ALL_FEATURE_COLUMNS)

label_encoder = fit_label_encoder(y_train_raw)
y_train = label_encoder.transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)

print(f'Train samples: {len(X_train_scaled)}, Test samples: {len(X_test_scaled)}')
print(f'Number of subject classes: {len(label_encoder.classes_)}')


## 3. Train and Compare Classical ML Classifiers
Models: Logistic Regression, K-Nearest Neighbors, Decision Tree, Random Forest, Support Vector Machine


In [ ]:
models, comparison_df, best_name = train_and_compare_all(
    X_train_scaled, y_train, X_test_scaled, y_test, seed=RANDOM_SEED
)
comparison_df


## 4. Visualize Model Comparison


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=comparison_df, x='Model', y='F1-Score (Macro)', palette='Blues_r')
plt.title('Classical ML Model Comparison (Macro F1-Score on Test Sessions 7-8)')
plt.ylabel('Macro F1-Score')
plt.ylim(0, 1.0)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 5. Best Classical Model Evaluation and Confusion Matrix


In [ ]:
best_model = models[best_name]
y_pred = best_model.predict(X_test_scaled)
metrics = compute_classification_metrics(y_test, y_pred)

print(f'Best Model: {best_name}')
print(f'Accuracy: {metrics["accuracy"]:.4f}')
print(f'Macro Precision: {metrics["precision_macro"]:.4f}')
print(f'Macro Recall: {metrics["recall_macro"]:.4f}')
print(f'Macro F1: {metrics["f1_macro"]:.4f}')

# Confusion matrix plot for sample classes
cm = np.array(metrics['confusion_matrix'])[:10, :10]
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_encoder.classes_[:10], yticklabels=label_encoder.classes_[:10])
plt.title(f'Confusion Matrix Sample (First 10 Subjects) - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()
